# Multi-Agent Workflow

Demonstrates a sequential multi-agent workflow where a **PlannerAgent** creates a story outline and a **WriterAgent** expands it into a full story.
Each agent runs as an independent Foundry persistent agent on its own thread.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
openai_client = client.get_openai_client()
print("Client ready.")

In [ ]:
# Create two specialized persistent Foundry v2 (Prompt) agents
planner = client.agents.create_version(
    agent_name="PlannerAgent",
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions="You create concise story outlines with 3-4 bullet points.",
    ),
)

writer = client.agents.create_version(
    agent_name="WriterAgent",
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions="You write short, engaging stories (3-4 paragraphs) based on outlines.",
    ),
)

print(f"PlannerAgent: {planner.name} (version {planner.version})")
print(f"WriterAgent:  {writer.name} (version {writer.version})")

In [ ]:
# Step 1 — Planner creates the story outline
print("=== PlannerAgent: Creating Story Outline ===\n")

response = openai_client.responses.create(
    input="Create an outline for a story about an AI that learns to paint.",
    extra_body={
        "agent_reference": {
            "name": planner.name,
            "version": planner.version,
            "type": "agent_reference",
        }
    },
)

outline = response.output_text
print(outline)

In [ ]:
# Step 2 — Writer expands the outline into a full story
print("\n=== WriterAgent: Writing the Story ===\n")

stream = openai_client.responses.create(
    stream=True,
    input=f"Write a story based on this outline:\n{outline}",
    extra_body={
        "agent_reference": {
            "name": writer.name,
            "version": writer.version,
            "type": "agent_reference",
        }
    },
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

In [ ]:
# Cleanup: delete both agents (all versions)
client.agents.delete(agent_name=planner.name)
client.agents.delete(agent_name=writer.name)
print("Both agents deleted.")